In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [1]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")

@wrap_model_call
def state_based_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Select model based on State conversation length."""
    message_count = len(request.messages)
    
    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model
        
    request = request.override(model=model)
    
    return handler(request)

In [3]:
from typing import List, Any

def print_messages(messages: List[Any]) -> None:
    for message in messages:
        message.pretty_print()

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Did you water the office plant today?")]
    }
)

print_messages(response["messages"])

================================ Human Message =================================

Did you water the office plant today?
================================== Ai Message ==================================

I can’t check in real time, but I haven’t watered it yet today. Want me to water it now? I can also set a daily reminder and log it in the plant care sheet if you’d like.


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?"),
            AIMessage(content="Yes, I gave it a light watering this morning."),
            HumanMessage(content="Has it grown much this week?"),
            AIMessage(content="It's sprouted two new leaves since Monday."),
            HumanMessage(content="Are the leaves still turning yellow on the edges?"),
            AIMessage(content="A little, but it's looking healthier overall."),
            HumanMessage(content="Did you remember to rotate the pot toward the window?"),
            AIMessage(content="I rotated it a quarter turn so it gets more even light."),
            HumanMessage(content="How often should we be fertilizing this plant?"),
            AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
            HumanMessage(content="When should we expect to have to replace the pot?")
        ]
    }
)

print_messages(response["messages"])

================================ Human Message =================================

Did you water the office plant today?
================================== Ai Message ==================================

Yes, I gave it a light watering this morning.
================================ Human Message =================================

Has it grown much this week?
================================== Ai Message ==================================

It's sprouted two new leaves since Monday.
================================ Human Message =================================

Are the leaves still turning yellow on the edges?
================================== Ai Message ==================================

A little, but it's looking healthier overall.
================================ Human Message =================================

Did you remember to rotate the pot toward the window?
================================== Ai Message ==================================

I rotated it a quarter turn so it gets

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-5-20250929


: 